# Turkish Legal RAG — Ablation Run

Runs the stage ablation on the default `turkish_legal_rag` eval set (195 article-labeled questions).

1. **Runtime → Change runtime type → T4 GPU** (or A100).
2. Run the cells in order. Every cell can be re-run on its own after a session restart.
3. Run the smoke test (step 6) before the full run (step 7).

In [ ]:
# 1. Clone the repo (combined_dataset.csv is stored with Git LFS)
import os
REPO = '/content/repo'
if not os.path.exists(REPO):
    !git lfs install
    !git clone -b improvements/2026-10 https://github.com/BeratMert29/Turkish-Legal-Question-Answering-RAG.git {REPO}
else:
    !git -C {REPO} pull --ff-only
os.chdir(f'{REPO}/CENG493_Project')
!git -C {REPO} log --oneline -1

In [ ]:
# 2. Install dependencies.
# If Colab shows a 'Restart session' prompt afterwards, accept it and continue with step 3.
import os
os.chdir('/content/repo/CENG493_Project')
!pip install -q -r requirements.txt

In [ ]:
# 3. Adapt config.py to the GPU. Edits the file itself so the !python subprocesses see it.
import os, re, subprocess
os.chdir('/content/repo/CENG493_Project')
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip()
LLM = 'qwen2.5:14b' if 'A100' in gpu else 'qwen2.5:7b'
JUDGE = 'llama3.1:8b'
BS = 64 if 'A100' in gpu else 16
s = open('config.py').read()
s = re.sub(r'^LLM_MODEL = .*$', f'LLM_MODEL = "{LLM}"', s, flags=re.M)
s = re.sub(r'^LLM_JUDGE_MODEL = .*$', f'LLM_JUDGE_MODEL = "{JUDGE}"', s, flags=re.M)
s = re.sub(r'^EMBEDDING_BATCH_SIZE = .*$', f'EMBEDDING_BATCH_SIZE = {BS}', s, flags=re.M)
open('config.py', 'w').write(s)
print(f'GPU: {gpu} | LLM: {LLM} | judge: {JUDGE} | batch: {BS}')
!grep -nE '^(LLM_MODEL|LLM_JUDGE_MODEL|EMBEDDING_BATCH_SIZE) =' config.py

In [ ]:
# 4. Install Ollama, start the server, pull the generator and judge models (~10 min)
import os, re, subprocess, time, shutil
os.chdir('/content/repo/CENG493_Project')
cfg = open('config.py').read()
LLM = re.search(r'^LLM_MODEL = "(.+)"', cfg, re.M).group(1)
JUDGE = re.search(r'^LLM_JUDGE_MODEL = "(.+)"', cfg, re.M).group(1)
if not shutil.which('ollama'):
    !curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
!ollama pull {LLM}
!ollama pull {JUDGE}
!ollama list

In [ ]:
# 5. Put the published index metadata where the graph stage looks for it
import os
os.chdir('/content/repo/CENG493_Project')
!mkdir -p index && cp ../results/index/* index/ && ls -la index/

In [ ]:
# 6. Smoke test: one stage, 10 questions (~5-10 min). If this fails, copy the error output.
import os
os.chdir('/content/repo/CENG493_Project')
!python scripts/14_eval_all_stages.py --stages base --limit 10

In [ ]:
# 7. Full ablation (~1-2 h on T4). Keep the tab open.
# emb_ft / full need fine-tuned weights that are not in the repo, so they are left out here.
import os
os.chdir('/content/repo/CENG493_Project')
!python scripts/14_eval_all_stages.py --stages base hybrid rrf rrf_rerank graph

In [ ]:
# 8. Download the results
import os, shutil
from google.colab import files
os.chdir('/content/repo/CENG493_Project')
shutil.make_archive('/content/ablation_results', 'zip', 'results')
files.download('/content/ablation_results.zip')